# EvalPlus benchmark: Qwen2.5-Coder-7B v4 (merged) — HumanEval/HumanEval+ và MBPP/MBPP+, prompt base và ChatML

Gộp `run-evalplus-kaggle.ipynb` (HumanEval) và `run-evalplus-mbpp.ipynb` (MBPP) thành một notebook. vLLM, greedy pass@1.
Mỗi model chạy 2 dataset × 2 kiểu prompt = 4 lần; mỗi lần EvalPlus báo cả bản gốc và bản `+` (HE, HE+, MBPP, MBPP+).

| Kiểu prompt | Cờ EvalPlus | Prompt gửi cho model |
|---|---|---|
| `base` | `--force-base-prompt` | completion thuần: signature + docstring, model viết tiếp |
| `chatml` | (không cờ) | chat template của tokenizer (ChatML, kèm system prompt mặc định của Qwen) |

**Trước khi chạy (Kaggle):**
1. Settings → Accelerator: **GPU T4 x2**. Settings → Internet: **On**.
2. Add-ons → Secrets: thêm `HF_TOKEN` (quyền đọc, repo `-merged` đang private) và attach vào notebook.

Model 7B fp16 (~15GB) không vừa 1 T4 nên dùng `--tp 2`; T4 không có bf16 nên dùng `float16`.

**Lưu ý quan trọng:** EvalPlus đặt tên file kết quả chỉ theo tên model, không theo kiểu prompt, và mặc định *resume* (dùng lại mẫu và điểm đã có). Chạy cùng model với hai kiểu prompt vào cùng thư mục thì lần sau sẽ lặp lại kết quả của lần trước. Notebook này tách thư mục kết quả theo kiểu prompt (`--root evalplus_results/<prompt>`) để tránh lỗi đó.

## 1. Cài đặt

In [ ]:
!pip install -q uv
!uv venv /kaggle/working/vllm_env --python 3.12
!uv pip install -q --python /kaggle/working/vllm_env/bin/python "evalplus[vllm]"
!/kaggle/working/vllm_env/bin/python -c "import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, '| vllm', vllm.__version__)"

## 2. Config

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# tag -> model. Thêm base để làm mốc: "base": "Qwen/Qwen2.5-Coder-7B"
MODELS = {
    "v4": "hoangnh39/qwen2.5-coder-7b-opencode-v4-merged",
}
DATASETS = ["humaneval", "mbpp"]   # mỗi dataset báo cả bản gốc và bản + (HE/HE+, MBPP/MBPP+)
PROMPTS = {                        # tên -> cờ EvalPlus thêm vào
    "base": "--force-base-prompt",
    "chatml": "",
}
TP, DTYPE = 2, "float16"
EVAL = "/kaggle/working/vllm_env/bin/evalplus.evaluate"

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Kiểm tra trước khi chạy

EvalPlus chỉ dùng chat template khi tokenizer của model có `chat_template`; thiếu thì kiểu `chatml` âm thầm chạy giống `base`. Cell này dừng sớm nếu thiếu, và in token dừng của model.

In [ ]:
from transformers import AutoTokenizer, GenerationConfig

for tag, model in MODELS.items():
    tok = AutoTokenizer.from_pretrained(model)
    assert tok.chat_template, f"{model}: tokenizer thiếu chat_template -> kiểu chatml sẽ chạy như base"
    print(f"{tag}: chat_template OK | generation_config eos = {GenerationConfig.from_pretrained(model).eos_token_id}")
    print(tok.apply_chat_template([{"role": "user", "content": "hi"}], tokenize=False, add_generation_prompt=True))

## 4. Chạy benchmark

Mỗi lần chạy là một tiến trình riêng nên GPU được giải phóng giữa các lần. Log: `evalplus_<tag>_<prompt>_<dataset>.log`; mẫu và kết quả chi tiết: `evalplus_results/<prompt>/<dataset>/`.

In [ ]:
for tag, model in MODELS.items():
    for ds in DATASETS:
        for prompt, flag in PROMPTS.items():
            print(f"===== {tag} / {ds} / {prompt} =====", flush=True)
            !{EVAL} --model {model} --dataset {ds} --backend vllm --greedy {flag} --root evalplus_results/{prompt} --tp {TP} --dtype {DTYPE} 2>&1 | tee evalplus_{tag}_{prompt}_{ds}.log | tail -n 12

## 5. Tổng hợp

In [ ]:
import re, glob
import pandas as pd

ANSI = re.compile(r"\x1b\[[0-9;]*m")
SCORE = re.compile(r"(humaneval|mbpp)(\+?) \((?:base tests|base \+ extra tests)\)\s*pass@1:\s*([\d.]+)")
COLS = {"humaneval": "HE", "humaneval+": "HE+", "mbpp": "MBPP", "mbpp+": "MBPP+"}

rows = {}
for path in sorted(glob.glob("evalplus_*_*_*.log")):
    tag, prompt, _ = re.fullmatch(r"evalplus_(.+?)_(base|chatml)_(humaneval|mbpp)\.log", path).groups()
    for name, plus, score in SCORE.findall(ANSI.sub("", open(path).read())):
        rows.setdefault((tag, prompt), {})[COLS[name + plus]] = round(float(score) * 100, 1)

table = pd.DataFrame(rows).T.reindex(columns=list(COLS.values()))
table.index.names = ["model", "prompt"]
table["avg"] = table.mean(axis=1).round(1)
print(table.to_string())
table.to_csv("evalplus_summary.csv")
try:
    print("\n" + table.to_markdown())   # cần tabulate
except ImportError:
    pass